# 公开 API 参数与调用示例

本 Notebook 按数据准备、分箱、编码、筛选、建模、评估、规则、报告和导出组织。
在项目根目录或 `examples/` 运行，使用本地 `hscredit_yyp.xlsx` 的完整 970 行验证数据。
该数据含业务信息，不随包发布；示例不依赖已执行的其他 Notebook。

目标 `FPD=1` 表示坏样本；分数方向必须按业务含义指定。统计、图形和报告均使用库内接口。
输出默认写入系统临时目录，可通过环境变量 `HSCREDIT_EXAMPLE_OUTPUT` 指定保留位置。末尾演示四类报告的独立导出与共享 writer，`sheet` 同时展示名称和 openpyxl 工作表对象。

In [1]:
from pathlib import Path
import os
from tempfile import mkdtemp
import numpy as np
import pandas as pd
import hscredit

root = Path.cwd() if (Path.cwd() / "examples").is_dir() else Path.cwd().parent
data = pd.read_excel(root / "examples" / "hscredit_yyp.xlsx")
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
X, y = data[features], data["FPD"]
output = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT") or mkdtemp(prefix="hscredit_api_reference_"))
output.mkdir(parents=True, exist_ok=True)
assert set(y.unique()) == {0, 1}
print(f"验证样本：{len(data)} 行，建模特征：{len(features)} 个")

验证样本：970 行，建模特征：3 个


## 分箱：方法枚举、输出枚举与未知类别

`OptimalBinning.VALID_METHODS` 是当前实现的方法集合。`transform` 的 `metric` 为
`indices`、`bins` 或 `woe`。训练后复用规则，数值切点属于右侧箱。
未知类别采用 `handle_unknown="value"` 时进入 `-3` 箱；`"raise"` 则拒绝未知类别。

In [2]:
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import compute_bin_stats

binner = OptimalBinning(method="quantile", max_n_bins=5, n_jobs=1).fit(X, y)
indices = binner.transform(X, metric="indices")
labels = binner.transform(X, metric="bins")
woe = binner.transform(X, metric="woe")
table = compute_bin_stats(indices[features[0]].to_numpy(), y.to_numpy())
assert int(table["样本总数"].sum()) == len(data)
assert indices.shape == labels.shape == woe.shape == X.shape
category_binner = OptimalBinning(method="quantile", handle_unknown="value", n_jobs=1)
category_binner.fit(data[["商品类别"]], y)
unknown = category_binner.transform(pd.DataFrame({"商品类别": ["文档测试未知类别"]}))
assert unknown.iloc[0, 0] == -3
print("分箱方法：", OptimalBinning.VALID_METHODS)

分箱方法： ['uniform', 'quantile', 'tree', 'chi', 'best_ks', 'best_iv', 'mdlp', 'or_tools', 'cp_sat', 'cart', 'kmeans', 'monotonic', 'genetic', 'smooth', 'kernel_density', 'best_lift', 'target_bad_rate']


## 编码和筛选

编码器的未知类别策略使用 `value/error/return_nan`，与分箱器的枚举不同。
筛选器通过 `get_support` 获取保留掩码；`SelectionReportCollector` 可采集已拟合筛选器。

In [3]:
from hscredit.core.encoders import WOEEncoder
from hscredit.core.selectors import VarianceSelector, SelectionReportCollector

encoder = WOEEncoder(features=["商品类别"], handle_unknown="value", n_jobs=1)
encoded = encoder.fit_transform(data[["商品类别"]], y)
assert len(encoded) == len(data)
selector = VarianceSelector(threshold=0.0, n_jobs=1).fit(woe, y)
selected = selector.transform(woe)
collector = SelectionReportCollector(source=selector)
selection_summary, selection_details = collector.report()
assert len(selector.get_support()) == len(features)
assert not selection_summary.empty and not selection_details.empty

## 建模与评估：在训练集拟合转换器

以下重新划分训练集和验证集，再仅使用训练集拟合分箱器。
预测的坏样本概率使用 `higher_risk`；若评估高分更安全的业务评分，应使用 `higher_safe`。

In [4]:
from sklearn.model_selection import train_test_split
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import auc, ks, psi

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
train_binner = OptimalBinning(method="quantile", max_n_bins=5, n_jobs=1).fit(X_train, y_train)
train_woe = train_binner.transform(X_train, metric="woe")
valid_woe = train_binner.transform(X_valid, metric="woe")
model = LogisticRegression(max_iter=200, calculate_stats=False, positive_woe_coef=False)
model.fit(train_woe, y_train)
prob = model.predict_proba(valid_woe)[:, 1]
evaluation = {"AUC": auc(y_valid, prob, score_direction="higher_risk"), "KS": ks(y_valid, prob),
              "PSI": psi(X_train[features[0]], X_valid[features[0]], method="quantile")}
assert all(np.isfinite(value) for value in evaluation.values())
print(evaluation)

{'AUC': 0.621609756097561, 'KS': 0.26136585365853654, 'PSI': 0.04104743882461182}


C:\Users\18306\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


## 规则和多标签报告

规则使用 `Rule`；金额统计通过 `amount` 指定。多标签入口使用
`overdue=["MOB1"]`、`dpds=[7, 3, 0]`，默认按 `>` 定义坏样本。
`target` 与 `overdue/dpds` 是两种不同的标签输入方式。

In [5]:
from hscredit.core.rules import Rule
from hscredit.report import feature_bin_stats, feature_group_binning_summary

rule = Rule("`衡枢鉴真分老客版` < 600")
rule_table = rule.report(data, target="FPD", amount="放款金额")
overdue_table = feature_bin_stats(data, feature=features[0], overdue=["MOB1"], dpds=[7, 3, 0],
                                method="quantile", amount="放款金额", long_format=True, n_jobs=1)
group_tables, group_summary = feature_group_binning_summary(
    data, feature=features[0], methods="quantile", target="FPD", date="放款时间", freq="M", n_jobs=1)
assert not rule_table.empty and not overdue_table.empty and not group_summary.empty

## EDA、金融计算、绘图与 Excel 导出

利率按期传入，支付现金流取负值。`bin_plot(result=True)` 返回 `(图对象, 统计表)`；
其他绘图函数的返回格式以各自文档为准。

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from hscredit.core.eda import data_info
from hscredit.core.financial import pmt, ipmt, ppmt
from hscredit.core.viz import bin_plot
from hscredit.excel import dataframe2excel

overview = data_info(data)
principal = float(data["放款金额"].dropna().iloc[0])
payment = pmt(0.12 / 12, 12, principal, when="end")
assert np.isclose(payment, ipmt(0.12 / 12, 1, 12, principal) + ppmt(0.12 / 12, 1, 12, principal))
figure, plot_table = bin_plot(data, target="FPD", feature=features[0], method="quantile",
                            max_n_bins=5, result=True, save=str(output / "分箱图.png"))
plt.close("all")
end_row, end_col = dataframe2excel(table, str(output / "分箱统计.xlsx"), sheet_name="分箱统计")
assert (output / "分箱图.png").is_file() and (output / "分箱统计.xlsx").is_file()
assert end_row > 2 and end_col > 2
print("数据概况、金融恒等式、图片和 Excel 导出验证通过")

数据概况、金融恒等式、图片和 Excel 导出验证通过


## 完整特征报告

`mode="strict"` 要求必需章节成功；`return_result=True` 返回结构化结果。
只需插入位置时，保留默认值可得到 `(end_row, end_col)`。

`save` 接收文件路径，`sheet` 指定工作表；传入 `save=writer` 则只写入，不自动保存或关闭。旧参数 `excel_path`、`excel_writer`、`sheet_name` 不再用于这些自动报告入口。底层 `dataframe2excel` 仍使用 `sheet_name`。

In [7]:
from hscredit.report import auto_feature_analysis

result = auto_feature_analysis(
    data, features=features, target="FPD", date="放款时间", amount="放款金额",
    save=output / "特征分析.xlsx", sheet="特征分析", output_dir=str(output / "图表"),
    pictures=["bin"], n_jobs=1, mode="strict", return_result=True,
)
assert result.metadata["输入行数"] == len(data)
assert (output / "特征分析.xlsx").is_file()
print("特征报告已保存")

特征计算:   0%|                    | 0/3 [       00:00<           ?,                ?字段/s]

特征报告已保存


## 模型、策略和树报告：使用相同的保存参数

下列调用继续使用完整样本。模型只在前文训练集上拟合；报告的特征列使用同一个已拟合分箱器转为 WOE，金额、日期与逾期天数保留原值。树同样只在训练集拟合，规则阈值也只从训练集确定。

四类报告都用 `save=路径` 直接保存；模型报告的 `sheet` 指定基本信息页，其他模型章节仍独立生成。

In [8]:
from hscredit.report import auto_model_report, swap_out_report
from hscredit.report.mining import MultiwayTreeExtractor

report_frames = {}
for name, indices in [("训练集", X_train.index), ("验证集", X_valid.index)]:
    frame = data.loc[indices].copy()
    frame[features] = train_binner.transform(frame[features], metric="woe")
    report_frames[name] = frame

report_labels = dict(overdue="MOB1", dpds=[7, 3, 0], overdue_operator=">", del_grey=True)
model_report = auto_model_report(
    model, report_frames, features=features, save=output / "模型报告.xlsx", sheet="模型基本信息",
    amount="放款金额", date="放款时间", long_format=True, with_plots=False,
    n_jobs=1, verbose=False, **report_labels,
)
threshold = float(data.loc[X_train.index, features[0]].quantile(0.25))
strategy_rule = Rule(f"`{features[0]}` < {threshold}", name="训练集低分段")
strategy_writer = swap_out_report(
    data.loc[X_valid.index], strategy_rule, save=output / "策略报告.xlsx", sheet="策略评估",
    amount="放款金额", date="放款时间", n_jobs=1, **report_labels,
)
tree = MultiwayTreeExtractor(target="FPD", features=features, max_depth=1, max_n_bins=3, n_jobs=1).fit(data.loc[X_train.index])
tree_frames = {"训练集": data.loc[X_train.index], "验证集": data.loc[X_valid.index]}
tree_reports = tree.report(
    tree_frames, save=output / "树报告.xlsx", sheet="规则评估",
    amount="放款金额", date="放款时间", methods=["quantile"], long_format=True, **report_labels,
)
assert set(tree_reports) == {"训练集", "验证集"}
assert all((output / name).is_file() for name in ["特征分析.xlsx", "模型报告.xlsx", "策略报告.xlsx", "树报告.xlsx"])
print("四类报告已按同一 save/sheet 约定保存")

四类报告已按同一 save/sheet 约定保存


## 共享 writer：同时使用名称与 Worksheet

`save=writer` 仅写入，不自动落盘。用不同的 `sheet` 承载不同报告，最后一次调用 `writer.save(...)`。传入的 Worksheet 必须属于该 writer 的工作簿。四类报告仍返回各自的报告对象、规则表或插入位置，不把它们当作文件路径。

In [9]:
from IPython.display import display
from hscredit.excel import ExcelWriter
from openpyxl import load_workbook

writer = ExcelWriter()
model_sheet = writer.get_sheet_by_name("模型基本信息")
merged_model = auto_model_report(
    model, report_frames, features=features, save=writer, sheet=model_sheet,
    amount="放款金额", long_format=True, with_plots=False, n_jobs=1, verbose=False, **report_labels,
)
feature_position = auto_feature_analysis(
    data, features, save=writer, sheet="特征分析", amount="放款金额", long_format=True,
    pictures=[], output_dir=str(output / "合并图表"), n_jobs=1, show_progress=False, **report_labels,
)
swap_out_report(data.loc[X_valid.index], strategy_rule, save=writer, sheet="策略评估", amount="放款金额", n_jobs=1, **report_labels)
tree.report(tree_frames, save=writer, sheet="规则评估", amount="放款金额", methods=["quantile"], **report_labels)
assert writer.workbook["模型基本信息"] is model_sheet
assert feature_position[0] > 2 and feature_position[1] > 2
combined_path = output / "综合报告.xlsx"
writer.save(combined_path)

workbook = load_workbook(combined_path, read_only=True)
try:
    expected_sheets = {"模型基本信息", "2-模型性能", "特征分析", "策略评估", "规则评估"}
    assert expected_sheets.issubset(workbook.sheetnames)
    display(pd.DataFrame({"工作表": workbook.sheetnames}))
finally:
    workbook.close()
print(f"公开 API 示例验证完成：{combined_path}")

,工作表
0,目录
1,模型基本信息
2,2-模型性能
3,3-入模变量分析
4,4-稳定性分析
5,5-模型参数
6,6-模型部署需求
7,7-训练与调参过程
8,特征分析
9,策略评估


公开 API 示例验证完成：C:\Users\18306\AppData\Local\Temp\hscredit-docs-29_api_reference-_ioeb8zf\reports\综合报告.xlsx
